# Solving Linear Systems with Message Passing

Most of us solve $Ax=b$ with a direct linear-algebra routine. In this tutorial we take a more scenic route: every unknown becomes a node in a graph, every nonzero off-diagonal entry becomes a coupling, and local Gaussian messages cooperate to recover the solution.

This is useful beyond solving one small system. The graph view exposes sparsity, supports distributed computation, and gives us a live view of convergence. It also reveals a subtle behavior: on graphs with cycles the means can be exact even when the reported variances are not.

In [ ]:
using LinearAlgebra
using Plots
using ReactiveMP
using RxInfer

default(; linewidth = 2, markersize = 5, legend = :best)

## A linear system as a Gaussian model

Assume that $A$ is symmetric and positive definite. The unnormalized Gaussian

$$p(x) \propto \exp\left(-\frac{1}{2}x^\mathsf{T}Ax+b^\mathsf{T}x\right)$$

has mean $A^{-1}b$, so finding its mean solves $Ax=b$. Expanding the exponent gives two kinds of factor:

- a Gaussian self-potential for each diagonal entry $A_{ii}$;
- an edge potential $\exp(-A_{ij}x_i x_j)$ for each nonzero off-diagonal entry.

The `Bilinear` node represents exactly that edge potential: `Bilinear(xᵢ, -A[i, j])`.

In [ ]:
@model function linear_system_model(b, A)
    n = length(b)

    # exp(bᵢxᵢ - Aᵢᵢxᵢ²/2), written as an ordinary normalized Gaussian.
    for i in 1:n
        x[i] ~ Normal(mean = b[i] / A[i, i], precision = A[i, i])
    end

    # One Bilinear factor for every edge of the sparse matrix.
    for i in 1:(n - 1), j in (i + 1):n
        if !iszero(A[i, j])
            x[j] ~ Bilinear(x[i], -A[i, j])
        end
    end
end

The graph can contain feedback loops, so it needs an initial message before updates can circulate. We start with a broad Gaussian message. This is only a computational starting point; it is not an extra prior in the model. `KeepEach()` records the belief after every sweep so that we can watch the algorithm settle.

In [ ]:
function solve_with_messages(A, b; iterations = 20)
    @assert issymmetric(A) "A must be symmetric"
    @assert isposdef(A) "A must be positive definite"

    initialization = @initialization begin
        μ(x) = NormalMeanVariance(0.0, 1e6)
    end

    result = infer(
        model = linear_system_model(A = A),
        data = (b = b,),
        initialization = initialization,
        returnvars = (x = KeepEach(),),
        iterations = iterations,
    )

    beliefs = result.posteriors[:x]
    means = reduce(hcat, (mean.(belief) for belief in beliefs))
    variances = reduce(hcat, (var.(belief) for belief in beliefs))
    return (; result, beliefs, means, variances)
end

## Warm-up: a chain

First consider four unknowns coupled in a chain. The corresponding factor graph has no cycles. This is the friendly case: both the means and marginal variances returned by message passing are exact.

In [ ]:
A_tree = [
    3.0  -0.8   0.0   0.0
   -0.8   3.5  -0.6   0.0
    0.0  -0.6   2.8  -0.7
    0.0   0.0  -0.7   2.5
]
b_tree = [1.0, -0.5, 2.0, 0.75]
tree = solve_with_messages(A_tree, b_tree; iterations = 12)

x_exact_tree = A_tree \ b_tree
v_exact_tree = diag(inv(A_tree))

println("largest mean error:     ", maximum(abs.(tree.means[:, end] - x_exact_tree)))
println("largest variance error: ", maximum(abs.(tree.variances[:, end] - v_exact_tree)))

In [ ]:
p_tree_graph = heatmap(
    abs.(A_tree .- Diagonal(diag(A_tree))),
    title = "Chain: edge strengths |Aᵢⱼ|",
    xlabel = "variable j", ylabel = "variable i",
    aspect_ratio = :equal, color = :blues, yflip = true, xticks = 1:4, yticks = 1:4,
)

p_tree_values = scatter(
    1:4, x_exact_tree, label = "A \\ b", marker = :circle,
    xlabel = "variable", ylabel = "solution", title = "Exact solution recovered",
)
scatter!(p_tree_values, 1:4, tree.means[:, end], label = "message passing", marker = :diamond)
plot(p_tree_graph, p_tree_values; layout = (1, 2), size = (900, 360))

## Add one edge: a loopy graph

Closing the chain into a ring creates a feedback loop. We keep the matrix strictly diagonally dominant, a convenient sufficient condition for convergence. Watch what changes: the means still converge to `A \ b`, while the variances become loop-dependent approximations.

In [ ]:
A_loop = copy(A_tree)
A_loop[1, 4] = A_loop[4, 1] = -0.9
loop = solve_with_messages(A_loop, b_tree; iterations = 20)

x_exact_loop = A_loop \ b_tree
v_exact_loop = diag(inv(A_loop))
residuals = [norm(A_loop * loop.means[:, k] - b_tree) for k in axes(loop.means, 2)]

@assert all(A_loop[i, i] > sum(abs, A_loop[i, :]) - A_loop[i, i] for i in axes(A_loop, 1))
println("final residual ‖Ax-b‖: ", residuals[end])

In [ ]:
p_convergence = plot(
    residuals, yscale = :log10, marker = :circle, label = false,
    xlabel = "message-passing sweep", ylabel = "‖Ax-b‖₂",
    title = "Local messages converge globally",
)

p_paths = plot(
    xlabel = "message-passing sweep", ylabel = "mean",
    title = "Belief means settling",
)
for i in axes(loop.means, 1)
    plot!(p_paths, loop.means[i, :], label = "x$i", marker = :circle)
    hline!(p_paths, [x_exact_loop[i]], color = i, linestyle = :dash, label = false)
end
plot(p_convergence, p_paths; layout = (1, 2), size = (950, 360))

In [ ]:
variance_error = 100 .* (loop.variances[:, end] .- v_exact_loop) ./ v_exact_loop

p_variances = bar(
    (1:4) .- 0.18, v_exact_loop,
    label = "diag(inv(A))", bar_width = 0.35,
    xlabel = "variable", ylabel = "marginal variance",
    title = "Cycles change the variance estimate", xticks = 1:4,
)
bar!(p_variances, (1:4) .+ 0.18, loop.variances[:, end], label = "loopy message passing", bar_width = 0.35)

p_variance_error = bar(
    1:4, variance_error, label = false,
    xlabel = "variable", ylabel = "relative error (%)",
    title = "Variance error despite exact means",
)
plot(p_variances, p_variance_error; layout = (1, 2), size = (950, 360))

## Turn up the coupling

Message passing is most comfortable when each diagonal term dominates the edges around it. The next experiment scales every off-diagonal coupling while keeping the diagonal fixed. As the system approaches the edge of diagonal dominance, information echoes more strongly around the ring and convergence slows.

In [ ]:
diagonal = Diagonal(diag(A_loop))
off_diagonal = A_loop - diagonal
coupling_scales = [0.25, 0.6, 0.9]

p_coupling = plot(
    yscale = :log10, xlabel = "message-passing sweep", ylabel = "‖Ax-b‖₂",
    title = "Stronger feedback needs more sweeps",
)
for scale in coupling_scales
    A_scaled = diagonal + scale * off_diagonal
    run = solve_with_messages(Matrix(A_scaled), b_tree; iterations = 20)
    errors = [norm(A_scaled * run.means[:, k] - b_tree) for k in axes(run.means, 2)]
    plot!(p_coupling, errors, marker = :circle, label = "scale = $scale")
end
p_coupling

## Takeaways

- `Bilinear(x[i], -A[i, j])` turns a matrix off-diagonal into a pairwise Gaussian coupling.
- Sparse matrices become sparse factor graphs, and local message updates recover the global solution.
- Loopy graphs need initial messages; `μ(x) = NormalMeanVariance(0, 1e6)` is a neutral broad starting point.
- Under suitable convergence conditions, the means solve $Ax=b$ exactly.
- Variances are exact on trees, but generally approximate on graphs with cycles.

For production linear solves, Julia's specialized factorizations remain the natural default. The message-passing formulation becomes interesting when the graph is distributed, when local updates matter, or when the linear system is one component of a larger probabilistic model.